# DQ Snapshot — Data Quality Metrics

Executes the combined DQ snapshot query against the **Bronze_Production_Lakehouse** SQL Analytics Endpoint
and appends aggregated results to the `data_quality_snapshot` Delta table.

**Sections:** Linkage Quality · Registry Parity · Completeness · Field Quality · Staleness · Match Readiness

**Output schema:** `snapshot_date`, `metric_category`, `metric_name`, `contact_type`,
`sales_decile`, `staleness_bucket`, `branch`, `creation_cohort`, `numerator`, `denominator`

**Schedule:** weekly · **Write mode:** `append` (snapshots stack for trend analysis)

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────
# Override via Fabric pipeline parameters or job definition.
SQL_ENDPOINT     = "your-workspace.datawarehouse.fabric.microsoft.com"
SOURCE_DATABASE  = "Bronze_Production_Lakehouse"
TARGET_LAKEHOUSE = "Silver_Analytics_Lakehouse"
TARGET_TABLE     = "data_quality_snapshot"
WRITE_MODE       = "append"   # "append" for weekly runs; "overwrite" to reset


In [ ]:
import struct
import datetime
import pyodbc
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from notebookutils import mssparkutils

spark = SparkSession.builder.getOrCreate()


In [ ]:
# ── Tier 1 — Revenue / Dimension Foundation ───────────────────────────────────
# dr: scalar date range — last day of last month back 59 months
_today     = datetime.date.today()
end_date   = _today.replace(day=1) - datetime.timedelta(days=1)
_em        = end_date.year * 12 + end_date.month - 59
start_date = datetime.date((_em - 1) // 12, (_em - 1) % 12 + 1, 1)

# Base table reads (reused across Tier 1 and below)
arm_cust        = spark.read.table("Bronze_Production_Lakehouse.Equip.ArMaster_Customer")
apmaster        = spark.read.table("Bronze_Production_Lakehouse.Equip.APMASTER")
vh_stock        = spark.read.table("Bronze_Production_Lakehouse.Equip.VhStock")
vh_stock_access = spark.read.table("Bronze_Production_Lakehouse.Equip.VhStockAccess")
invoice         = spark.read.table("Bronze_Production_Lakehouse.Equip.Invoice")
rental_hist     = spark.read.table("Bronze_Production_Lakehouse.Equip.Rental_History")

# VhStockAccess: pre-aggregate Sale_Value per Stock_No (resolves correlated subquery in CompleteGoods)
_vsa_agg = (
    vh_stock_access
    .groupBy("Stock_No")
    .agg(F.coalesce(F.sum(F.coalesce(F.col("Sale_Value"), F.lit(0))), F.lit(0)).alias("vsa_total"))
)

# Filtered invoice base (invo_type IN ('C','I')) — reused for Parts, Service, Rental, last_tx
_inv_base = invoice.filter(F.col("invo_type").isin("C", "I"))

# ── account_revenue ────────────────────────────────────────────────────────────
# CompleteGoods: equipment sales from VhStock + VhStockAccess
_cg = (
    arm_cust.alias("am")
    .join(
        vh_stock.filter(F.col("SALESDATE").between(F.lit(start_date), F.lit(end_date))).alias("vhs"),
        F.col("vhs.Owner") == F.col("am.contact_code"),
    )
    .join(_vsa_agg.alias("vsa"), F.col("vhs.NO") == F.col("vsa.Stock_No"), "left")
    .groupBy(F.col("vhs.NO"), F.col("am.Customer_No"))
    .agg(
        F.round(
            F.coalesce(F.sum(F.coalesce(F.col("vhs.SALES_VALUE"), F.lit(0))), F.lit(0))
            + F.coalesce(F.first(F.col("vsa.vsa_total")), F.lit(0)),
            2,
        ).cast("decimal(12,2)").alias("CompleteGoods")
    )
    .select(
        F.col("Customer_No").alias("AccountNumber"),
        F.col("CompleteGoods"),
        F.lit(0).cast("decimal(12,2)").alias("Parts"),
        F.lit(0).cast("decimal(12,2)").alias("Service"),
        F.lit(0).cast("decimal(12,2)").alias("Rental"),
    )
)

# Parts: Invoice module_type='I'
_parts = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter(
            (F.col("module_type") == "I") &
            F.col("invo_datetime").between(F.lit(start_date), F.lit(end_date))
        ).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .groupBy(F.col("am.Customer_No"))
    .agg(
        F.round(F.sum(F.coalesce(F.col("i.parts_sale_val"), F.lit(0))), 2)
        .cast("decimal(12,2)").alias("Parts")
    )
    .select(
        F.col("Customer_No").alias("AccountNumber"),
        F.lit(0).cast("decimal(12,2)").alias("CompleteGoods"),
        F.col("Parts"),
        F.lit(0).cast("decimal(12,2)").alias("Service"),
        F.lit(0).cast("decimal(12,2)").alias("Rental"),
    )
)

# Service: Invoice module_type='W'
_service = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter(
            (F.col("module_type") == "W") &
            F.col("invo_datetime").between(F.lit(start_date), F.lit(end_date))
        ).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .groupBy(F.col("am.Customer_No"))
    .agg(
        F.round(F.sum(
            F.coalesce(F.col("i.parts_sale_val"),  F.lit(0)) +
            F.coalesce(F.col("i.labour_sale_val"), F.lit(0)) +
            F.coalesce(F.col("i.sublet_sal_val"),  F.lit(0)) +
            F.coalesce(F.col("i.other_sale_val"),  F.lit(0))
        ), 2).cast("decimal(12,2)").alias("Service")
    )
    .select(
        F.col("Customer_No").alias("AccountNumber"),
        F.lit(0).cast("decimal(12,2)").alias("CompleteGoods"),
        F.lit(0).cast("decimal(12,2)").alias("Parts"),
        F.col("Service"),
        F.lit(0).cast("decimal(12,2)").alias("Rental"),
    )
)

# Rental: Invoice (no module_type filter) LEFT JOIN Rental_History
_rental = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter(F.col("invo_datetime").between(F.lit(start_date), F.lit(end_date))).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .join(rental_hist.alias("rh"), F.col("i.document_no") == F.col("rh.Invoice_No"), "left")
    .groupBy(F.col("am.Customer_No"))
    .agg(
        F.round(F.sum(F.coalesce(F.col("rh.Value"), F.lit(0))), 2)
        .cast("decimal(12,2)").alias("Rental")
    )
    .select(
        F.col("Customer_No").alias("AccountNumber"),
        F.lit(0).cast("decimal(12,2)").alias("CompleteGoods"),
        F.lit(0).cast("decimal(12,2)").alias("Parts"),
        F.lit(0).cast("decimal(12,2)").alias("Service"),
        F.col("Rental"),
    )
)

account_revenue = (
    _cg.unionByName(_parts).unionByName(_service).unionByName(_rental)
    .groupBy("AccountNumber")
    .agg(F.sum(F.col("CompleteGoods") + F.col("Parts") + F.col("Service") + F.col("Rental")).alias("TotalRevenue"))
)

# revenue_ranked: decile by cumulative revenue share — D1 = accounts making up top 10% of total $
# Uses running cumulative sum ordered by revenue DESC; each account's position determines its decile.
_rev_win_cum   = Window.orderBy(F.desc("TotalRevenue")).rowsBetween(Window.unboundedPreceding, Window.currentRow)
_rev_total_win = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

revenue_ranked = (
    account_revenue
    .filter(F.col("TotalRevenue") > 0)
    .withColumn("cumrev",    F.sum("TotalRevenue").over(_rev_win_cum))
    .withColumn("total_rev", F.sum("TotalRevenue").over(_rev_total_win))
    .withColumn("RawDecile", F.least(F.ceil(F.col("cumrev") / F.col("total_rev") * 10).cast("int"), F.lit(10)))
    .select("AccountNumber", "RawDecile")
)

# last_tx: MAX transaction date per account across all four transaction streams
_lt1 = (
    arm_cust.alias("am")
    .join(
        vh_stock.filter(F.col("SALESDATE").isNotNull()).alias("vhs"),
        F.col("vhs.Owner") == F.col("am.contact_code"),
    )
    .select(F.col("am.Customer_No").alias("acc_no"), F.col("vhs.SALESDATE").cast("date").alias("tx_date"))
)
_lt2 = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter((F.col("module_type") == "I") & F.col("invo_datetime").isNotNull()).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .select(F.col("am.Customer_No").alias("acc_no"), F.col("i.invo_datetime").cast("date").alias("tx_date"))
)
_lt3 = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter((F.col("module_type") == "W") & F.col("invo_datetime").isNotNull()).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .select(F.col("am.Customer_No").alias("acc_no"), F.col("i.invo_datetime").cast("date").alias("tx_date"))
)
_lt4 = (
    arm_cust.alias("am")
    .join(
        _inv_base.filter(F.col("invo_datetime").isNotNull()).alias("i"),
        F.col("i.bill_to_acc") == F.col("am.Customer_No"),
    )
    .join(rental_hist.alias("rh"), F.col("i.document_no") == F.col("rh.Invoice_No"))
    .select(F.col("am.Customer_No").alias("acc_no"), F.col("i.invo_datetime").cast("date").alias("tx_date"))
)

last_tx = (
    _lt1.unionByName(_lt2).unionByName(_lt3).unionByName(_lt4)
    .groupBy("acc_no")
    .agg(F.max("tx_date").alias("last_tx_date"))
)

print(f"Tier 1 built: date range {start_date} – {end_date}")
print("DataFrames: account_revenue, revenue_ranked, last_tx")


In [ ]:
# ── Tier 1 Summary — Revenue by Sales Decile ──────────────────────────────────
sales_by_decile = (
    account_revenue
    .join(revenue_ranked, "AccountNumber", "left")
    .groupBy("RawDecile")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.sum("TotalRevenue"), 2).alias("total_revenue"),
    )
    .withColumn(
        "sales_decile",
        F.when(F.col("RawDecile").isNull(), "Unranked")
         .otherwise(F.concat(F.lit("D"), F.col("RawDecile").cast("string")))
    )
    .orderBy(F.coalesce(F.col("RawDecile"), F.lit(99)))
    .select("sales_decile", "customer_count", "total_revenue")
)

sales_by_decile.toPandas()


In [ ]:
# ── Tier 2 — Master Contact Base ──────────────────────────────────────────────
contact_tbl = spark.read.table("Bronze_Production_Lakehouse.Equip.contact")
wkmech      = spark.read.table("Bronze_Production_Lakehouse.Equip.WKMECHFL")
vh_salman   = spark.read.table("Bronze_Production_Lakehouse.Equip.VhSalman")

def _nullif_trim(col_expr):
    """NULLIF(LTRIM(RTRIM(col)), '') — blank strings become null."""
    return F.when(F.trim(col_expr) == "", None).otherwise(F.trim(col_expr))

# active_contacts: non-inactive, non-employee contacts with account info
active_contacts = (
    contact_tbl.alias("c")
    .join(wkmech.select(F.col("Code").alias("wk_code")).alias("wk"),
          F.col("c.contact_code") == F.col("wk.wk_code"), "left")
    .join(vh_salman.select(F.col("CODE").alias("vs_code")).alias("vs"),
          F.col("c.contact_code") == F.col("vs.vs_code"), "left")
    .join(arm_cust.alias("am"), F.col("am.contact_code") == F.col("c.contact_code"), "left")
    .join(apmaster.select(F.col("CONTACT_CODE").alias("ap_code")).alias("ap"),
          F.col("c.contact_code") == F.col("ap.ap_code"), "left")
    .filter(
        (F.coalesce(F.col("c.Inactive_Indicator"), F.lit("A")) != "I") &
        F.col("wk.wk_code").isNull() &
        F.col("vs.vs_code").isNull() &
        ~(F.col("am.contact_code").isNull() & F.col("ap.ap_code").isNotNull())
    )
    .select(
        F.col("c.contact_code"),
        F.col("c.Business_Individual"),
        F.col("c.Ckc_Id"),
        F.col("c.Creation_Date"),
        _nullif_trim(F.col("c.name")).alias("first_name"),
        _nullif_trim(F.col("c.surname")).alias("last_name"),
        _nullif_trim(F.col("c.company_name")).alias("company_name"),
        _nullif_trim(F.col("c.email_address")).alias("email"),
        _nullif_trim(F.col("c.BusinessPhone")).alias("biz_phone"),
        _nullif_trim(F.col("c.PrivatePhone")).alias("priv_phone"),
        _nullif_trim(F.col("c.MobilePhone")).alias("mob_phone"),
        _nullif_trim(F.col("c.street")).alias("street"),
        _nullif_trim(F.col("c.city")).alias("city"),
        _nullif_trim(F.col("c.state")).alias("state"),
        _nullif_trim(F.col("c.pcode")).alias("pcode"),
        _nullif_trim(F.col("c.country")).alias("country"),  # no US default here
        _nullif_trim(F.col("c.title")).alias("title"),
        _nullif_trim(F.col("c.Generation")).alias("generation"),
        _nullif_trim(F.col("c.Suffix")).alias("suffix"),
        F.col("am.Customer_No").alias("acc_no"),
        _nullif_trim(F.col("am.TERRITORY")).alias("branch"),
    )
)

# contact_enriched: adds staleness_bucket, sales_decile, creation_cohort
_today_lit = F.current_date()

contact_enriched = (
    active_contacts.alias("ac")
    .join(last_tx.alias("lt"), F.col("ac.acc_no") == F.col("lt.acc_no"), "left")
    .join(revenue_ranked.alias("rr"), F.col("ac.acc_no") == F.col("rr.AccountNumber"), "left")
    .select(
        F.col("ac.*"),
        F.when(F.col("ac.acc_no").isNull(), "no_account")
         .when(F.col("lt.last_tx_date").isNull(), "never_transacted")
         .when(F.col("lt.last_tx_date") >= F.date_add(_today_lit, -365), "0-1yr")
         .when(F.col("lt.last_tx_date") >= F.date_add(_today_lit, -730), "1-2yr")
         .when(F.col("lt.last_tx_date") >= F.date_add(_today_lit, -1095), "2-3yr")
         .when(F.col("lt.last_tx_date") >= F.date_add(_today_lit, -1460), "3-4yr")
         .when(F.col("lt.last_tx_date") >= F.date_add(_today_lit, -1825), "4-5yr")
         .otherwise("5+yr")
         .alias("staleness_bucket"),
        F.when(F.col("ac.acc_no").isNull(), "Unranked")
         .when(F.col("rr.AccountNumber").isNull(), "Unranked")
         .otherwise(F.concat(F.lit("D"), F.col("rr.RawDecile").cast("string")))
         .alias("sales_decile"),
        F.when(F.col("ac.Creation_Date").isNull(), "Unknown")
         .when(F.year(F.col("ac.Creation_Date")) < 2016, "Pre-2015")
         .when(F.year(F.col("ac.Creation_Date")) <= 2020, "2016-2020")
         .when(F.year(F.col("ac.Creation_Date")) <= 2025, "2021-2025")
         .otherwise("2026+")
         .alias("creation_cohort"),
    )
)

contact_enriched = contact_enriched.cache()
print("Tier 2 built: active_contacts, contact_enriched")


In [ ]:
# ── Tier 3a — Population Subsets: contact_linkage, unlinked_enriched, active_linked, inactive_linked
cross_ref   = spark.read.table("Bronze_Production_Lakehouse.DDP.customer_cross_ref")
cust_profile = spark.read.table("Bronze_Production_Lakehouse.DDP.customer_profile")

_xr_active = cross_ref.filter(F.col("entity_id") != 999999998)
_xr_hutson  = _xr_active.filter(F.col("cross_ref_description") == "HUTSON INC Dealer XREF")

# contact_linkage: LEFT JOIN filtered to Hutson cross-refs only
contact_linkage = (
    contact_enriched.alias("ce")
    .join(
        _xr_hutson.alias("xr"),
        F.upper(F.col("xr.cross_ref_number")) == F.upper(F.col("ce.contact_code")),
        "left",
    )
    .select(
        F.col("ce.contact_code"),
        F.col("ce.Business_Individual"),
        F.col("ce.Ckc_Id"),
        F.col("ce.sales_decile"),
        F.col("ce.staleness_bucket"),
        F.col("ce.branch"),
        F.col("ce.creation_cohort"),
        F.col("xr.entity_id"),
        F.col("xr.contact_id"),
        F.when(F.col("xr.cross_ref_number").isNotNull(), 1).otherwise(0).alias("is_linked"),
    )
)

# unlinked_enriched: contacts with no cross_ref entry (left anti-join)
_xr_keys = _xr_hutson.select(F.upper(F.col("cross_ref_number")).alias("_norm"))
unlinked_enriched = contact_enriched.join(
    _xr_keys, F.upper(contact_enriched["contact_code"]) == _xr_keys["_norm"], "left_anti"
).cache()

# active_linked: active contacts with HUTSON XREF + customer_profile
active_linked = (
    contact_enriched.alias("ce")
    .join(
        _xr_hutson.alias("xr"),
        F.upper(F.col("xr.cross_ref_number")) == F.upper(F.col("ce.contact_code")),
    )
    .join(
        cust_profile.alias("cp"),
        (F.col("cp.entity_id") == F.col("xr.entity_id")) &
        (F.col("cp.contact_id") == F.col("xr.contact_id")),
    )
    .select(
        F.col("ce.contact_code"),
        F.col("ce.Business_Individual"),
        F.col("ce.sales_decile"),
        F.col("ce.staleness_bucket"),
        F.col("ce.branch"),
        F.col("ce.creation_cohort"),
        F.col("ce.company_name"),
        F.col("ce.first_name"),
        F.col("ce.last_name"),
        F.col("ce.email"),
        F.col("ce.biz_phone"),
        F.col("ce.priv_phone"),
        F.col("ce.mob_phone"),
        F.col("ce.street"),
        F.col("ce.city"),
        F.col("ce.state"),
        F.col("ce.pcode"),
        F.col("ce.country").alias("country"),
        _nullif_trim(F.col("cp.nm1_txt")).alias("reg_company_name"),
        _nullif_trim(F.col("cp.first_nm")).alias("reg_first_name"),
        _nullif_trim(F.col("cp.last_nm")).alias("reg_last_name"),
        _nullif_trim(F.col("cp.email_addr_txt")).alias("reg_email"),
        _nullif_trim(F.col("cp.work_area_cd")).alias("reg_biz_area"),
        _nullif_trim(F.col("cp.work_phone_num")).alias("reg_biz_num"),
        _nullif_trim(F.col("cp.home_area_cd")).alias("reg_priv_area"),
        _nullif_trim(F.col("cp.home_phone_num")).alias("reg_priv_num"),
        _nullif_trim(F.col("cp.mobile_area_cd")).alias("reg_mob_area"),
        _nullif_trim(F.col("cp.mobile_phone_num")).alias("reg_mob_num"),
        _nullif_trim(F.col("cp.phys_street1_txt")).alias("reg_street"),
        _nullif_trim(F.col("cp.phys_city")).alias("reg_city"),
        _nullif_trim(F.col("cp.phys_state_prov_cd")).alias("reg_state"),
        _nullif_trim(F.col("cp.phys_postal_cd")).alias("reg_pcode"),
        _nullif_trim(F.col("cp.phys_iso2_cntry_cd")).alias("reg_country"),
        F.col("cp.phys_postal_certified"),
        F.col("cp.phys_undeliverable_ind"),
        F.col("cp.mail_undeliverable_ind"),
        F.col("cp.out_of_busn_ind"),
        F.col("cp.descd_ind"),
    )
)

# inactive_linked: Inactive_Indicator='I', non-employee, linked via HUTSON XREF
inactive_linked = (
    contact_tbl.alias("c")
    .join(wkmech.select(F.col("Code").alias("wk_code")).alias("wk"),
          F.col("c.contact_code") == F.col("wk.wk_code"), "left")
    .join(vh_salman.select(F.col("CODE").alias("vs_code")).alias("vs"),
          F.col("c.contact_code") == F.col("vs.vs_code"), "left")
    .join(arm_cust.alias("am"), F.col("am.contact_code") == F.col("c.contact_code"), "left")
    .join(apmaster.select(F.col("CONTACT_CODE").alias("ap_code")).alias("ap"),
          F.col("c.contact_code") == F.col("ap.ap_code"), "left")
    .join(
        _xr_hutson.alias("xr"),
        F.upper(F.col("xr.cross_ref_number")) == F.upper(F.col("c.contact_code")),
    )
    .join(
        cust_profile.alias("cp"),
        (F.col("cp.entity_id") == F.col("xr.entity_id")) &
        (F.col("cp.contact_id") == F.col("xr.contact_id")),
    )
    .filter(
        (F.col("c.Inactive_Indicator") == "I") &
        F.col("wk.wk_code").isNull() &
        F.col("vs.vs_code").isNull() &
        ~(F.col("am.contact_code").isNull() & F.col("ap.ap_code").isNotNull())
    )
    .select(
        F.col("c.contact_code"),
        F.col("c.Business_Individual"),
        _nullif_trim(F.col("c.Inactive_Reason")).alias("inactive_reason"),
        F.col("cp.out_of_busn_ind"),
        F.col("cp.descd_ind"),
        F.when(F.col("c.Creation_Date").isNull(), "Unknown")
         .when(F.year(F.col("c.Creation_Date")) < 2016, "Pre-2015")
         .when(F.year(F.col("c.Creation_Date")) <= 2020, "2016-2020")
         .when(F.year(F.col("c.Creation_Date")) <= 2025, "2021-2025")
         .otherwise("2026+").alias("creation_cohort"),
        _nullif_trim(F.col("am.TERRITORY")).alias("branch"),
        F.lit("Inactive").alias("sales_decile"),
        F.lit("Inactive").alias("staleness_bucket"),
    )
)

print("Tier 3a built: contact_linkage, unlinked_enriched, active_linked, inactive_linked")


In [ ]:
# ── Tier 3b — field_parity, linked_counts, dup_codes ─────────────────────────
_dim = ["sales_decile", "staleness_bucket", "branch", "creation_cohort"]

def _parity_case(eq, rg):
    """Standard case-insensitive parity (UPPER both sides)."""
    return (
        F.when(eq.isNull() & rg.isNull(), "both_null")
         .when(eq.isNotNull() & rg.isNull(), "equip_only")
         .when(eq.isNull() & rg.isNotNull(), "registry_only")
         .when(F.upper(eq) == F.upper(rg), "match")
         .otherwise("mismatch")
    )

def _phone_parity(phone, area, num):
    """Parity for split area-code + number registry fields."""
    reg_combined = F.coalesce(area, F.lit("")) + F.coalesce(num, F.lit(""))
    reg_null     = area.isNull() & num.isNull()
    reg_notnull  = area.isNotNull() | num.isNotNull()
    return (
        F.when(phone.isNull() & reg_null, "both_null")
         .when(phone.isNotNull() & reg_null, "equip_only")
         .when(phone.isNull() & reg_notnull, "registry_only")
         .when(phone == reg_combined, "match")
         .otherwise("mismatch")
    )

def _fp(al, field_lit, contact_type_col, parity_col, extra_filter=None):
    df = al if extra_filter is None else al.filter(extra_filter)
    return df.select(
        F.lit(field_lit).alias("field_name"),
        contact_type_col.alias("contact_type"),
        *[F.col(c) for c in _dim],
        parity_col.alias("parity_result"),
    )

al = active_linked  # shorthand

_fp_parts = [
    _fp(al, "company_name", F.col("Business_Individual"),
        _parity_case(F.col("company_name"), F.col("reg_company_name")),
        F.col("Business_Individual").isin("B", "C")),

    _fp(al, "first_name", F.col("Business_Individual"),
        _parity_case(F.col("first_name"), F.col("reg_first_name")),
        F.col("Business_Individual").isin("I", "C")),

    _fp(al, "last_name", F.col("Business_Individual"),
        _parity_case(F.col("last_name"), F.col("reg_last_name")),
        F.col("Business_Individual").isin("I", "C")),

    _fp(al, "email", F.col("Business_Individual"),
        _parity_case(F.col("email"), F.col("reg_email"))),  # UPPER is fine for email match

    _fp(al, "business_phone", F.col("Business_Individual"),
        _phone_parity(F.col("biz_phone"), F.col("reg_biz_area"), F.col("reg_biz_num"))),

    _fp(al, "private_phone", F.col("Business_Individual"),
        _phone_parity(F.col("priv_phone"), F.col("reg_priv_area"), F.col("reg_priv_num"))),

    _fp(al, "mobile_phone", F.col("Business_Individual"),
        _phone_parity(F.col("mob_phone"), F.col("reg_mob_area"), F.col("reg_mob_num"))),

    _fp(al, "street", F.col("Business_Individual"),
        _parity_case(F.col("street"), F.col("reg_street"))),

    _fp(al, "city", F.col("Business_Individual"),
        _parity_case(F.col("city"), F.col("reg_city"))),

    _fp(al, "state", F.col("Business_Individual"),
        _parity_case(F.col("state"), F.col("reg_state"))),

    _fp(al, "zip", F.col("Business_Individual"),
        # US: normalize both sides to first 5 digits; non-US: raw compare
        F.when(
            F.coalesce(F.col("country"), F.lit("US")) == "US",
            _parity_case(F.col("pcode").substr(1, 5), F.col("reg_pcode").substr(1, 5)),
        ).otherwise(
            _parity_case(F.col("pcode"), F.col("reg_pcode"))
        )),

    _fp(al, "country", F.col("Business_Individual"),
        _parity_case(F.col("country"), F.col("reg_country"))),
]

field_parity = _fp_parts[0]
for _part in _fp_parts[1:]:
    field_parity = field_parity.unionByName(_part)

# linked_counts: count and certified count per dimension slice
linked_counts = (
    active_linked
    .groupBy("Business_Individual", *_dim)
    .agg(
        F.count("*").alias("total_linked"),
        F.sum(F.when(F.col("phys_postal_certified") == "CERTIFIED", 1).otherwise(0)).alias("certified_linked"),
    )
)

# dup_codes: normalized contact_codes that appear more than once
dup_codes = (
    contact_enriched
    .groupBy(F.upper(F.trim(F.col("contact_code"))).alias("norm_code"))
    .agg(F.count("*").alias("_n"))
    .filter(F.col("_n") > 1)
    .select("norm_code")
)

field_parity = field_parity.cache()
print("Tier 3b built: field_parity, linked_counts, dup_codes")


In [ ]:
# ── Section 1 — Linkage Quality ───────────────────────────────────────────────
_snap        = F.current_date()
_lq_dims     = ["Business_Individual", "sales_decile", "staleness_bucket", "branch", "creation_cohort"]
_out_cols    = ["snapshot_date", "metric_category", "metric_name", "contact_type",
                "sales_decile", "staleness_bucket", "branch", "creation_cohort",
                "numerator", "denominator"]

def _lq_select(df, metric_name, num_col, den_col):
    return df.select(
        _snap.alias("snapshot_date"),
        F.lit("linkage").alias("metric_category"),
        F.lit(metric_name).alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col(num_col).cast("long").alias("numerator"),
        F.col(den_col).cast("long").alias("denominator"),
    )

# Metrics 1–3: single pass over contact_linkage
_s1_agg = (
    contact_linkage
    .groupBy(*_lq_dims)
    .agg(
        F.sum("is_linked").alias("linked"),
        F.sum(F.lit(1) - F.col("is_linked")).alias("unlinked"),
        F.count("*").alias("total"),
        F.sum(F.when(F.col("Ckc_Id").isNotNull() & (F.col("is_linked") == 0), 1).otherwise(0)).alias("ckc_no_xref_n"),
        F.sum(F.when(F.col("Ckc_Id").isNotNull(), 1).otherwise(0)).alias("ckc_total"),
    )
)

_s1_linked_count    = _lq_select(_s1_agg, "linked_count",       "linked",       "total")
_s1_unlinked_count  = _lq_select(_s1_agg, "unlinked_count",     "unlinked",     "total")
_s1_ckc_no_xref     = _lq_select(_s1_agg, "ckc_id_no_cross_ref","ckc_no_xref_n","ckc_total")

# Metric 4: type_mismatch_linkage — linked contacts only
_s1_type_mismatch = (
    contact_linkage.filter(F.col("is_linked") == 1)
    .groupBy(*_lq_dims)
    .agg(
        F.sum(
            F.when(
                (F.col("Business_Individual") == "C") &
                (F.col("contact_id").isNull() | (F.col("contact_id") == 0)), 1)
             .when(
                F.col("Business_Individual").isin("B", "I") &
                F.col("contact_id").isNotNull() & (F.col("contact_id") != 0), 1)
             .otherwise(0)
        ).cast("long").alias("numerator"),
        F.sum("is_linked").cast("long").alias("denominator"),
    )
    .select(
        _snap.alias("snapshot_date"), F.lit("linkage").alias("metric_category"),
        F.lit("type_mismatch_linkage").alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

# Metric 5: duplicate_entity_id — ALL dims; uses crossJoin for denominator
_linked_cl = contact_linkage.filter(F.col("is_linked") == 1)
_s1_dup_eid = (
    _linked_cl.groupBy("entity_id").agg(F.count("*").alias("_n")).filter(F.col("_n") > 1)
    .agg(F.count("*").cast("long").alias("numerator"))
    .crossJoin(_linked_cl.select("entity_id").distinct().agg(F.count("*").cast("long").alias("denominator")))
    .select(
        _snap.alias("snapshot_date"), F.lit("linkage").alias("metric_category"),
        F.lit("duplicate_entity_id").alias("metric_name"),
        F.lit("ALL").alias("contact_type"), F.lit("ALL").alias("sales_decile"),
        F.lit("ALL").alias("staleness_bucket"), F.lit(None).cast("string").alias("branch"),
        F.lit("ALL").alias("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

# Metric 6: orphan_cross_ref — Hutson cross_ref rows with no matching contact_enriched
_xr_hutson_codes = _xr_hutson.select(F.col("cross_ref_number"))
_ce_upper = contact_enriched.select(F.upper(F.col("contact_code")).alias("_norm"))
_orphans = (
    _xr_hutson_codes
    .join(_ce_upper, F.upper(F.col("cross_ref_number")) == F.col("_norm"), "left_anti")
)
_s1_orphan = (
    _orphans.agg(F.count("*").cast("long").alias("numerator"))
    .crossJoin(_xr_hutson_codes.agg(F.count("*").cast("long").alias("denominator")))
    .select(
        _snap.alias("snapshot_date"), F.lit("linkage").alias("metric_category"),
        F.lit("orphan_cross_ref").alias("metric_name"),
        F.lit("ALL").alias("contact_type"), F.lit("ALL").alias("sales_decile"),
        F.lit("ALL").alias("staleness_bucket"), F.lit(None).cast("string").alias("branch"),
        F.lit("ALL").alias("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

section_1 = (
    _s1_linked_count
    .unionByName(_s1_unlinked_count)
    .unionByName(_s1_ckc_no_xref)
    .unionByName(_s1_type_mismatch)
    .unionByName(_s1_dup_eid)
    .unionByName(_s1_orphan)
)

print("Section 1 built: 6 linkage quality metrics")


In [ ]:
# ── Section 2 — Registry Parity ───────────────────────────────────────────────
_parity_dims = ["contact_type", "sales_decile", "staleness_bucket", "branch", "creation_cohort"]

# Field-level parity rows: spine approach — all 5 outcomes emitted per dim slice,
# numerator=0 when no contacts land in that outcome (e.g. first_name_registry_only).
_outcomes_sdf = spark.createDataFrame(
    [("match",), ("mismatch",), ("equip_only",), ("registry_only",), ("both_null",)],
    ["parity_result"],
)
# Denominator: total in-scope contacts per (field, contact_type, dim slice)
_fp_den = (
    field_parity
    .groupBy("field_name", "contact_type", *_dim)
    .agg(F.count("*").cast("long").alias("denominator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)
# Actual counts (only outcomes that occurred)
_fp_actual = (
    field_parity
    .groupBy("field_name", "parity_result", "contact_type", *_dim)
    .agg(F.count("*").cast("long").alias("numerator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)
# Spine: all dim slices × all 5 outcomes; left join fills missing outcomes with 0
_fp_agg = (
    _fp_den
    .crossJoin(_outcomes_sdf)
    .join(_fp_actual, on=["field_name", "parity_result", "contact_type"] + _dim, how="left")
    .withColumn("numerator", F.coalesce(F.col("numerator"), F.lit(0)).cast("long"))
    .withColumn("branch", F.when(F.col("branch") == "", None).otherwise(F.col("branch")))
)
_s2_field_rows = (
    _fp_agg
    .select(
        _snap.alias("snapshot_date"),
        F.lit("parity").alias("metric_category"),
        # F.concat required — PySpark + on string Columns does numeric addition, returning null for non-numeric strings
        F.concat(F.col("field_name"), F.lit("_"), F.col("parity_result")).alias("metric_name"),
        F.col("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

# active_linked × linked_counts — reused for all priority metrics
_al_lc = (
    active_linked.alias("al")
    .join(
        linked_counts.alias("lc"),
        (F.col("al.Business_Individual") == F.col("lc.Business_Individual")) &
        (F.col("al.sales_decile")         == F.col("lc.sales_decile")) &
        (F.col("al.staleness_bucket")     == F.col("lc.staleness_bucket")) &
        (F.coalesce(F.col("al.branch"), F.lit("")) == F.coalesce(F.col("lc.branch"), F.lit(""))) &
        (F.col("al.creation_cohort")      == F.col("lc.creation_cohort")),
    )
)

_parity_group_cols = [F.col("al.Business_Individual"), F.col("al.sales_decile"),
                      F.col("al.staleness_bucket"), F.col("al.branch"), F.col("al.creation_cohort")]

def _parity_priority_select(df, metric_name):
    return df.select(
        _snap.alias("snapshot_date"), F.lit("parity").alias("metric_category"),
        F.lit(metric_name).alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )

# phys_addr_certified_mismatch: certified address differs between EQUIP and Registry.
# Groups ALL linked contacts first so denominator rows always exist for groups with certified
# contacts — the original filter-first approach produced no rows when numerator was 0.
_s2_cert_mismatch = _parity_priority_select(
    _al_lc
    .groupBy(*_parity_group_cols)
    .agg(
        F.sum(
            F.when(
                (F.col("al.phys_postal_certified") == "CERTIFIED") &
                (
                    (F.upper(F.coalesce(F.col("al.street"), F.lit(""))) != F.upper(F.coalesce(F.col("al.reg_street"), F.lit("")))) |
                    (F.upper(F.coalesce(F.col("al.city"),   F.lit(""))) != F.upper(F.coalesce(F.col("al.reg_city"),   F.lit("")))) |
                    (F.upper(F.coalesce(F.col("al.state"),  F.lit(""))) != F.upper(F.coalesce(F.col("al.reg_state"),  F.lit("")))) |
                    (F.upper(F.coalesce(F.col("al.pcode"),  F.lit(""))) != F.upper(F.coalesce(F.col("al.reg_pcode"),  F.lit(""))))
                ),
                1,
            ).otherwise(0)
        ).cast("long").alias("numerator"),
        F.max(F.col("lc.certified_linked")).cast("long").alias("denominator"),
    )
    .filter(F.col("denominator") > 0),
    "phys_addr_certified_mismatch",
)

# address_confirmed_undeliverable
_s2_undeliverable = _parity_priority_select(
    _al_lc
    .filter((F.col("al.phys_undeliverable_ind") == "Y") | (F.col("al.mail_undeliverable_ind") == "Y"))
    .groupBy(*_parity_group_cols)
    .agg(F.count("*").cast("long").alias("numerator"),
         F.max(F.col("lc.total_linked")).cast("long").alias("denominator")),
    "address_confirmed_undeliverable",
)

# registry_oob_equip_active: Business type, Registry shows OOB but EQUIP is active
_s2_oob = _parity_priority_select(
    _al_lc
    .filter((F.col("al.Business_Individual") == "B") & (F.col("al.out_of_busn_ind") == "Y"))
    .groupBy(*_parity_group_cols)
    .agg(F.count("*").cast("long").alias("numerator"),
         F.max(F.col("lc.total_linked")).cast("long").alias("denominator")),
    "registry_oob_equip_active",
)

# registry_deceased_equip_active: Individual/Contact type, Registry shows deceased
_s2_deceased = _parity_priority_select(
    _al_lc
    .filter(F.col("al.Business_Individual").isin("I", "C") & (F.col("al.descd_ind") == "Y"))
    .groupBy(*_parity_group_cols)
    .agg(F.count("*").cast("long").alias("numerator"),
         F.max(F.col("lc.total_linked")).cast("long").alias("denominator")),
    "registry_deceased_equip_active",
)

# equip_inactive_reason_mismatch: inactive contacts where EQUIP reason ≠ Registry flags
_s2_inactive_mismatch = (
    inactive_linked
    .groupBy("Business_Individual", "sales_decile", "staleness_bucket", "branch", "creation_cohort")
    .agg(
        F.sum(
            F.when(
                (F.col("inactive_reason") == "Out of Business") &
                (F.coalesce(F.col("out_of_busn_ind"), F.lit("N")) != "Y"), 1
            ).when(
                (F.col("inactive_reason") == "Deceased") &
                (F.coalesce(F.col("descd_ind"), F.lit("N")) != "Y"), 1
            ).otherwise(0)
        ).cast("long").alias("numerator"),
        F.count("*").cast("long").alias("denominator"),
    )
    .select(
        _snap.alias("snapshot_date"), F.lit("parity").alias("metric_category"),
        F.lit("equip_inactive_reason_mismatch").alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

section_2 = (
    _s2_field_rows
    .unionByName(_s2_cert_mismatch)
    .unionByName(_s2_undeliverable)
    .unionByName(_s2_oob)
    .unionByName(_s2_deceased)
    .unionByName(_s2_inactive_mismatch)
)

print("Section 2 built: field-level parity + 5 priority metrics")


In [ ]:
# ── Section 3 — Completeness ──────────────────────────────────────────────────
_ce = contact_enriched  # shorthand

def _comp_row(metric, issue_col, extra_filter=None):
    df = _ce if extra_filter is None else _ce.filter(extra_filter)
    return df.select(
        F.lit(metric).alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        issue_col.alias("is_issue"),
    )

_s3_parts = [
    _comp_row("missing_first_name",   F.when(F.col("first_name").isNull(),   1).otherwise(0), F.col("Business_Individual").isin("I", "C")),
    _comp_row("missing_last_name",    F.when(F.col("last_name").isNull(),    1).otherwise(0), F.col("Business_Individual").isin("I", "C")),
    _comp_row("missing_company_name", F.when(F.col("company_name").isNull(), 1).otherwise(0), F.col("Business_Individual").isin("B", "C")),
    _comp_row("missing_street",   F.when(F.col("street").isNull(),   1).otherwise(0)),
    _comp_row("missing_city",     F.when(F.col("city").isNull(),     1).otherwise(0)),
    _comp_row("missing_state",    F.when(F.col("state").isNull(),    1).otherwise(0)),
    _comp_row("missing_zip",      F.when(F.col("pcode").isNull(),    1).otherwise(0)),
    _comp_row("missing_country",  F.when(F.col("country").isNull(),  1).otherwise(0)),
    _comp_row("missing_email",    F.when(F.col("email").isNull(),    1).otherwise(0)),
    _comp_row("missing_all_phones",
        F.when(F.col("biz_phone").isNull() & F.col("priv_phone").isNull() & F.col("mob_phone").isNull(), 1).otherwise(0)),
    _comp_row("no_contact_info",
        F.when(F.col("biz_phone").isNull() & F.col("priv_phone").isNull() & F.col("mob_phone").isNull() & F.col("email").isNull(), 1).otherwise(0)),
]

_s3_union = _s3_parts[0]
for _p in _s3_parts[1:]:
    _s3_union = _s3_union.unionByName(_p)

section_3 = (
    _s3_union
    .groupBy("metric_name", "contact_type", "sales_decile", "staleness_bucket", "branch", "creation_cohort")
    .agg(F.sum("is_issue").cast("long").alias("numerator"), F.count("*").cast("long").alias("denominator"))
    .select(
        _snap.alias("snapshot_date"), F.lit("completeness").alias("metric_category"),
        F.col("metric_name"), F.col("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

print("Section 3 built: 11 completeness metrics")


In [ ]:
# ── Section 4 — Field Quality ─────────────────────────────────────────────────
# Helper: all characters in a string are the same
def _all_same(c):
    return F.length(F.regexp_replace(c, "^(.)\\1*$", "")) == 0

def _fq_row(metric, issue_col, extra_filter=None):
    df = _ce if extra_filter is None else _ce.filter(extra_filter)
    return df.select(
        F.lit(metric).alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        issue_col.alias("is_issue"),
    )

_ic = F.col("Business_Individual").isin("I", "C")  # shorthand filter

_s4_parts = [
    # ── Name: placeholder values ──────────────────────────────────────────────
    _fq_row("placeholder_name",
        F.when(
            F.upper(F.col("first_name")).isin("FIRSTNAME","FIRST NAME","FIRST","FNAME","N/A","NA","NONE","UNKNOWN","NOT APPLICABLE","BLANK") |
            F.upper(F.col("last_name")).isin("LASTNAME","LAST NAME","LAST","LNAME","N/A","NA","NONE","UNKNOWN","NOT APPLICABLE","BLANK"), 1).otherwise(0),
        _ic),

    # ── Company name: placeholder values (B/C) ───────────────────────────────────────
    _fq_row("placeholder_company_name",
        F.when(
            F.upper(F.col("company_name")).isin("COMPANY","COMPANY NAME","N/A","NA","NONE","UNKNOWN","NOT APPLICABLE","BLANK","NO NAME","NAME"), 1).otherwise(0),
        F.col("Business_Individual").isin("B", "C")),

    # ── Name: all same character ──────────────────────────────────────────────
    # ── Name: all same character (I/C) ──────────────────────────────────────────────
    _fq_row("name_all_same_char",
        F.when((F.col("first_name").isNotNull() & _all_same(F.col("first_name"))) |
               (F.col("last_name").isNotNull() & _all_same(F.col("last_name"))), 1).otherwise(0),
        _ic),

    # ── Company name: all same character (B/C) ───────────────────────────────────────
    _fq_row("company_name_all_same_char",
        F.when(F.col("company_name").isNotNull() & _all_same(F.col("company_name")), 1).otherwise(0),
        F.col("Business_Individual").isin("B", "C")),

    # ── Name: numeric only ────────────────────────────────────────────────────
    # ── Name: numeric only (I/C) ─────────────────────────────────────────────────────
    _fq_row("name_numeric_only",
        F.when((F.col("first_name").isNotNull() & F.col("first_name").rlike("^[0-9]+$")) |
               (F.col("last_name").isNotNull() & F.col("last_name").rlike("^[0-9]+$")), 1).otherwise(0),
        _ic),

    # ── Company name: numeric only (B/C) ─────────────────────────────────────────────
    _fq_row("company_name_numeric_only",
        F.when(F.col("company_name").isNotNull() & F.col("company_name").rlike("^[0-9]+$"), 1).otherwise(0),
        F.col("Business_Individual").isin("B", "C")),

    # ── Name: status/alert text in first or last name ─────────────────────────
    _fq_row("status_text_in_name",
        F.when(
            F.upper(F.col("first_name")).rlike("DECEASED|OUT OF BUSINESS|DO NOT USE|DONT USE|DON'T USE| USE |INACTIVE|CLOSED|FARM PLAN|BAD DEBT|WRITE OFF|DO NOT SELL|DO NOT SERVE") |
            F.upper(F.col("last_name")).rlike("DECEASED|OUT OF BUSINESS|DO NOT USE|DONT USE|DON'T USE| USE |INACTIVE|CLOSED|FARM PLAN|BAD DEBT|WRITE OFF|DO NOT SELL|DO NOT SERVE"),
            1).otherwise(0),
        _ic),

    # ── Name: status/alert text in company name ───────────────────────────────
    _fq_row("status_text_in_company",
        F.when(
            F.upper(F.col("company_name")).rlike("DECEASED|OUT OF BUSINESS|DO NOT USE|DONT USE|DON'T USE| USE |INACTIVE|CLOSED|FARM PLAN| OOB |BAD DEBT|WRITE OFF|DO NOT SELL|DO NOT SERVE"),
            1).otherwise(0),
        F.col("Business_Individual").isin("B", "C")),

    # ── Address: status/alert text in street ──────────────────────────────────
    _fq_row("status_text_in_street",
        F.when(
            F.upper(F.col("street")).rlike("DECEASED|OUT OF BUSINESS|DO NOT USE|DONT USE|DON'T USE|INACTIVE|CLOSED|BAD DEBT|WRITE OFF|DO NOT SELL|DO NOT SERVE"),
            1).otherwise(0)),

    # ── Address: placeholder street ───────────────────────────────────────────
    _fq_row("placeholder_street",
        F.when(F.upper(F.col("street")).isin("N/A","NA","NONE","UNKNOWN","UNK","TBD","NO ADDRESS","ADDRESS","NO STREET","-"), 1).otherwise(0),
        F.col("street").isNotNull()),

    # ── Address: placeholder city ─────────────────────────────────────────────
    _fq_row("placeholder_city",
        F.when(F.upper(F.col("city")).isin("N/A","NA","NONE","UNKNOWN","UNK","TBD","NO CITY","CITY","-"), 1).otherwise(0),
        F.col("city").isNotNull()),

    # ── Address: placeholder state ────────────────────────────────────────────
    _fq_row("placeholder_state",
        F.when(F.upper(F.col("state")).isin("N/A","NA","NONE","UNKNOWN","UNK","XX","ZZ"), 1).otherwise(0),
        F.col("state").isNotNull()),

    # ── Company: DBA pattern ──────────────────────────────────────────────────
    _fq_row("dba_in_company_name",
        F.when(
            F.upper(F.col("company_name")).contains("DBA ") |
            F.upper(F.col("company_name")).contains("D/B/A") |
            F.upper(F.col("company_name")).contains("DOING BUSINESS AS"),
            1).otherwise(0),
        F.col("Business_Individual").isin("B", "C")),

    # ── Test / dummy records ──────────────────────────────────────────────────
    _fq_row("test_record",
        F.when(
            (F.col("first_name").isNotNull() & F.upper(F.col("first_name")).isin("TEST","TESTING","TEMP","DUMMY","SAMPLE")) |
            (F.col("last_name").isNotNull() & F.upper(F.col("last_name")).isin("TEST","TESTING","TEMP","DUMMY","SAMPLE")) |
            (F.col("company_name").isNotNull() & F.upper(F.col("company_name")).isin("TEST","TESTING","DUMMY","SAMPLE","TEMP")),
            1).otherwise(0)),

    # ── Contact type / field mismatch ─────────────────────────────────────────
    _fq_row("contact_type_field_mismatch",
        F.when(
            ((F.col("Business_Individual") == "B") & (F.col("first_name").isNotNull() | F.col("last_name").isNotNull()) & F.col("company_name").isNull()) |
            (_ic & F.col("company_name").isNotNull() & F.col("first_name").isNull() & F.col("last_name").isNull()),
            1).otherwise(0)),

    # ── Name: prefix in first_name ────────────────────────────────────────────
    _fq_row("prefix_in_name",
        F.when(
            F.upper(F.col("first_name")).rlike("^(MR|MRS|MS|DR|REV|PROF)\\.") |
            F.upper(F.col("first_name")).rlike("^(MR|MRS|MS|DR|REV|PROF) "),
            1).otherwise(0),
        _ic),

    # ── Name: suffix in last_name ─────────────────────────────────────────────
    _fq_row("suffix_in_surname",
        F.when(
            F.upper(F.col("last_name")).rlike(" (JR|JR\\.|SR|SR\\.|II|III|IV|V|MD|PHD|CPA|ESQ|DDS|DO)$"),
            1).otherwise(0),
        _ic),

    # ── Name: combined names in first_name ────────────────────────────────────
    _fq_row("combined_names_in_name",
        F.when(
            F.col("first_name").contains("&") |
            F.upper(F.col("first_name")).contains(" AND ") |
            F.col("first_name").contains("/") |
            F.upper(F.col("first_name")).contains(" OR "),
            1).otherwise(0),
        _ic),

    # ── Name: familiar name pattern "(Nick)" ──────────────────────────────────
    _fq_row("familiar_name_pattern",
        F.when(F.col("first_name").contains("("), 1).otherwise(0),
        _ic),

    # ── Email: invalid format ─────────────────────────────────────────────────
    _fq_row("email_invalid_format",
        F.when(F.col("email").isNotNull() & ~F.col("email").rlike("@.+\\..+"), 1).otherwise(0)),

    # ── Email: placeholder domain ─────────────────────────────────────────────
    _fq_row("email_placeholder",
        F.when(
            F.lower(F.col("email")).rlike("^(noemail|nomail|donotcontact|noreply)@") |
            F.lower(F.col("email")).startswith("test@test") |
            F.lower(F.col("email")).startswith("none@none"),
            1).otherwise(0),
        F.col("email").isNotNull()),

    # ── Phone: biz ───────────────────────────────────────────────────────────
    _fq_row("biz_phone_sequential",     F.when(F.col("biz_phone") == "1234567890", 1).otherwise(0), F.col("biz_phone").isNotNull()),
    _fq_row("biz_phone_repeated_digit", F.when((F.length(F.col("biz_phone")) == 10) & _all_same(F.col("biz_phone")), 1).otherwise(0), F.col("biz_phone").isNotNull()),
    _fq_row("biz_phone_wrong_length",   F.when(~F.length(F.col("biz_phone")).isin(10, 11), 1).otherwise(0), F.col("biz_phone").isNotNull()),

    # ── Phone: priv ──────────────────────────────────────────────────────────
    _fq_row("priv_phone_sequential",     F.when(F.col("priv_phone") == "1234567890", 1).otherwise(0), F.col("priv_phone").isNotNull()),
    _fq_row("priv_phone_repeated_digit", F.when((F.length(F.col("priv_phone")) == 10) & _all_same(F.col("priv_phone")), 1).otherwise(0), F.col("priv_phone").isNotNull()),
    _fq_row("priv_phone_wrong_length",   F.when(~F.length(F.col("priv_phone")).isin(10, 11), 1).otherwise(0), F.col("priv_phone").isNotNull()),

    # ── Phone: mob ───────────────────────────────────────────────────────────
    _fq_row("mob_phone_sequential",     F.when(F.col("mob_phone") == "1234567890", 1).otherwise(0), F.col("mob_phone").isNotNull()),
    _fq_row("mob_phone_repeated_digit", F.when((F.length(F.col("mob_phone")) == 10) & _all_same(F.col("mob_phone")), 1).otherwise(0), F.col("mob_phone").isNotNull()),
    _fq_row("mob_phone_wrong_length",   F.when(~F.length(F.col("mob_phone")).isin(10, 11), 1).otherwise(0), F.col("mob_phone").isNotNull()),

    # ── Address: field length checks ──────────────────────────────────────────
    _fq_row("state_not_2char",   F.when(F.length(F.col("state")) != 2, 1).otherwise(0), F.col("state").isNotNull()),
    _fq_row("country_not_2char", F.when(F.length(F.col("country")) != 2, 1).otherwise(0), F.col("country").isNotNull()),

    # ── Address: zip not 5-digit (US only) ───────────────────────────────────
    _fq_row("zip_invalid_format",
        F.when(
            ~F.col("pcode").rlike("^[0-9]{5}$") &
            ~F.col("pcode").rlike("^[0-9]{5}-[0-9]{4}$") &
            ~F.col("pcode").rlike("^[0-9]{9}$"),
            1).otherwise(0),
        F.col("pcode").isNotNull() & (F.coalesce(F.col("country"), F.lit("US")) == "US")),

    # ── Coded fields: generation ──────────────────────────────────────────────
    _fq_row("generation_unrecognized",
        F.when(~F.upper(F.trim(F.col("generation"))).isin("JR","JR.","JUNIOR","SR","SR.","SENIOR","II","III","IV","V"), 1).otherwise(0),
        _ic & F.col("generation").isNotNull()),

    # ── Coded fields: title ───────────────────────────────────────────────────
    _fq_row("title_unrecognized",
        F.when(~F.upper(F.trim(F.col("title"))).isin(
            "MR","MR.","MRS","MRS.","MS","MS.","MISS",
            "DR","DR.","REV","REV.","PROF","PROF.",
            "CAPT","CAPT.","SGT","SGT.","COL","COL.",
            "MAJ","MAJ.","GEN","GEN.","HON","HON."), 1).otherwise(0),
        _ic & F.col("title").isNotNull()),

    # ── Coded fields: suffix ──────────────────────────────────────────────────
    _fq_row("suffix_unrecognized",
        F.when(~F.upper(F.trim(F.col("suffix"))).isin(
            "MD","DO","DDS","DMD","DVM","PHD","PH.D.","JD","CPA",
            "RN","NP","PA","PE","ESQ","ESQ.","MBA","CFA","CFP",
            "LCSW","CPCU","FACP","FACS"), 1).otherwise(0),
        _ic & F.col("suffix").isNotNull()),

    # ── Branch: invalid/closed code ───────────────────────────────────────────
    _fq_row("invalid_branch",
        F.when(F.col("branch").isin("07","13","52","55","61","63","64","67","70","71"), 1).otherwise(0),
        F.col("branch").isNotNull()),

    # ── contact_code: whitespace ──────────────────────────────────────────────
    _fq_row("contact_code_has_whitespace",
        F.when(F.col("contact_code").rlike("[ \\t\\n\\r]"), 1).otherwise(0)),

    # ── contact_code: non-alphanumeric ────────────────────────────────────────
    _fq_row("contact_code_non_alphanumeric",
        F.when(F.col("contact_code").rlike("[^A-Za-z0-9]"), 1).otherwise(0)),
]

_s4_union = _s4_parts[0]
for _p in _s4_parts[1:]:
    _s4_union = _s4_union.unionByName(_p)

_s4_main = (
    _s4_union
    .groupBy("metric_name", "contact_type", "sales_decile", "staleness_bucket", "branch", "creation_cohort")
    .agg(F.sum("is_issue").cast("long").alias("numerator"), F.count("*").cast("long").alias("denominator"))
    .select(
        _snap.alias("snapshot_date"), F.lit("field_quality").alias("metric_category"),
        F.col("metric_name"), F.col("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

# contact_code_duplicate_normalized: left join so every dim slice emits a row;
# numerator=0 when no normalized duplicates exist; denominator = contacts in that slice.
_s4_dup_codes = (
    contact_enriched
    .join(dup_codes, F.upper(F.trim(F.col("contact_code"))) == dup_codes["norm_code"], "left")
    .withColumn("_is_dup", F.when(F.col("norm_code").isNotNull(), 1).otherwise(0))
    .groupBy("Business_Individual", "sales_decile", "staleness_bucket", "branch", "creation_cohort")
    .agg(
        F.sum("_is_dup").cast("long").alias("numerator"),
        F.count("*").cast("long").alias("denominator"),
    )
    .select(
        _snap.alias("snapshot_date"), F.lit("field_quality").alias("metric_category"),
        F.lit("contact_code_duplicate_normalized").alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"), F.col("staleness_bucket"),
        F.col("branch"), F.col("creation_cohort"),
        F.col("numerator"), F.col("denominator"),
    )
)

section_4 = _s4_main.unionByName(_s4_dup_codes)

print("Section 4 built: 40 field quality metrics")


In [ ]:
# ── Section 5 — Staleness ─────────────────────────────────────────────────────
# metric_name IS the staleness_bucket; output staleness_bucket column = 'ALL'
# Spine: all 8 buckets × every dim slice so denominators are uniform when Power BI
# rolls up — missing buckets (no contacts in that slice) get numerator = 0.
_s5_dims = ["Business_Individual", "sales_decile", "branch", "creation_cohort"]

_staleness_buckets = spark.createDataFrame(
    [("no_account",), ("never_transacted",), ("0-1yr",), ("1-2yr",),
     ("2-3yr",), ("3-4yr",), ("4-5yr",), ("5+yr",)],
    ["staleness_bucket"],
)

# Denominator: total contacts per dim slice (same for all 8 buckets in that slice)
# Branch coalesced to "" before joining — NULL == NULL is false in PySpark equi-join.
_s5_den = (
    contact_enriched
    .groupBy(*_s5_dims)
    .agg(F.count("*").cast("long").alias("denominator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)

# Actual counts per bucket × dim slice
_s5_actual = (
    contact_enriched
    .groupBy("staleness_bucket", *_s5_dims)
    .agg(F.count("*").cast("long").alias("numerator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)

# Spine: all dim slices × all 8 buckets; left join fills missing buckets with 0;
# restore branch NULL after join.
section_5 = (
    _s5_den
    .crossJoin(_staleness_buckets)
    .join(_s5_actual, on=["staleness_bucket"] + _s5_dims, how="left")
    .withColumn("numerator", F.coalesce(F.col("numerator"), F.lit(0)).cast("long"))
    .withColumn("branch", F.when(F.col("branch") == "", None).otherwise(F.col("branch")))
    .select(
        _snap.alias("snapshot_date"),
        F.lit("staleness").alias("metric_category"),
        F.col("staleness_bucket").alias("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"),
        F.lit("ALL").alias("staleness_bucket"),
        F.col("branch"),
        F.col("creation_cohort"),
        F.col("numerator"),
        F.col("denominator"),
    )
)

# ── Section 6 — Match Readiness ───────────────────────────────────────────────
# Scope: unlinked active non-employee contacts only.
# Tier 4 = no usable name; Tier 1 = name + full/near-full address;
# Tier 2 = name + partial address or any contact info; Tier 3 = name only.
# Spine: all 4 tiers × every dim slice so denominators are uniform when Power BI
# rolls up — missing tiers (no contacts in that slice) get numerator = 0.
_s6_dims = ["Business_Individual", "sales_decile", "staleness_bucket", "branch", "creation_cohort"]

_s6_tiered = (
    unlinked_enriched
    .withColumn("tier",
        F.when(
            (F.col("Business_Individual").isin("I","C") & (F.col("first_name").isNull() | F.col("last_name").isNull())) |
            ((F.col("Business_Individual") == "B") & F.col("company_name").isNull()),
            4,
        ).when(
            (F.col("street").isNotNull() & F.col("city").isNotNull() & F.col("state").isNotNull()) |
            (F.col("street").isNotNull() & F.col("pcode").isNotNull()),
            1,
        ).when(
            F.col("street").isNotNull() | F.col("city").isNotNull() |
            F.col("state").isNotNull() | F.col("pcode").isNotNull() |
            F.col("biz_phone").isNotNull() | F.col("priv_phone").isNotNull() |
            F.col("mob_phone").isNotNull() | F.col("email").isNotNull(),
            2,
        ).otherwise(3)
    )
)

_tiers_sdf = spark.createDataFrame(
    [(1, "tier_1_strong"), (2, "tier_2_partial"), (3, "tier_3_name_only"), (4, "tier_4_no_name")],
    ["tier", "metric_name"],
)

# Denominator: total unlinked contacts per dim slice (same for all 4 tiers in that slice)
# Branch coalesced to "" before joining — NULL == NULL is false in PySpark equi-join.
_s6_den = (
    unlinked_enriched
    .groupBy(*_s6_dims)
    .agg(F.count("*").cast("long").alias("denominator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)

# Actual counts per tier × dim slice
_s6_actual = (
    _s6_tiered
    .groupBy("tier", *_s6_dims)
    .agg(F.count("*").cast("long").alias("numerator"))
    .withColumn("branch", F.coalesce(F.col("branch"), F.lit("")))
)

# Spine: all dim slices × all 4 tiers; left join fills missing tiers with 0;
# restore branch NULL after join.
section_6 = (
    _s6_den
    .crossJoin(_tiers_sdf)
    .join(_s6_actual, on=["tier"] + _s6_dims, how="left")
    .withColumn("numerator", F.coalesce(F.col("numerator"), F.lit(0)).cast("long"))
    .withColumn("branch", F.when(F.col("branch") == "", None).otherwise(F.col("branch")))
    .select(
        _snap.alias("snapshot_date"),
        F.lit("match_readiness").alias("metric_category"),
        F.col("metric_name"),
        F.col("Business_Individual").alias("contact_type"),
        F.col("sales_decile"),
        F.col("staleness_bucket"),
        F.col("branch"),
        F.col("creation_cohort"),
        F.col("numerator"),
        F.col("denominator"),
    )
)

print("Sections 5 & 6 built: staleness distribution + match readiness tiers")


In [ ]:
# ── Final Assembly — union all six sections into sdf for write ─────────────────
sdf = (
    section_1
    .unionByName(section_2)
    .unionByName(section_3)
    .unionByName(section_4)
    .unionByName(section_5)
    .unionByName(section_6)
)

sdf.printSchema()
sdf.groupBy("metric_category").count().orderBy("metric_category").show(truncate=False)
print(f"Total rows to write: {sdf.count():,}")


In [ ]:
# ── Write to Delta table ──────────────────────────────────────────────────────
(
    sdf.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("mergeSchema", "true")
    .saveAsTable(f"{TARGET_LAKEHOUSE}.{TARGET_TABLE}")
)
print(f"Written to {TARGET_LAKEHOUSE}.{TARGET_TABLE} (mode={WRITE_MODE})")

# ── Verification: row counts for the snapshot just written ────────────────────
spark.sql(f"""
    SELECT
        snapshot_date,
        metric_category,
        COUNT(*)         AS row_count,
        SUM(numerator)   AS total_numerator,
        SUM(denominator) AS total_denominator
    FROM {TARGET_LAKEHOUSE}.{TARGET_TABLE}
    WHERE snapshot_date = (
        SELECT MAX(snapshot_date)
        FROM {TARGET_LAKEHOUSE}.{TARGET_TABLE}
    )
    GROUP BY snapshot_date, metric_category
    ORDER BY metric_category
""").show(truncate=False)


In [ ]:
# ── Metric Dimension Table ─────────────────────────────────────────────────────
# One row per metric_name — source of truth for Power BI display names, definitions,
# and issue-rate filtering (Action Items 1 & 2, dq-review-notes.md).
# Overwritten on each run; this is a dim table, not a time series.
DIM_TABLE = "data_quality_metric_dim"

_rows = []

def _r(metric_name, category, label, definition, is_issue):
    _rows.append(dict(
        metric_name=metric_name,
        metric_category=category,
        label=label,
        definition=definition,
        is_issue=is_issue,
    ))

# ── Linkage Quality ────────────────────────────────────────────────────────────
_r("linked_count", "linkage", "Linked Contacts",
   "Active contacts matched to the Registry via customer_cross_ref. "
   "Numerator: active non-employee contacts with a cross_ref entry (sentinel 999999998 excluded). "
   "Denominator: all active non-employee contacts.",
   False)

_r("unlinked_count", "linkage", "Unlinked Contacts",
   "Active contacts with no matching customer_cross_ref entry. "
   "Numerator: contacts absent from cross_ref. "
   "Denominator: all active non-employee contacts.",
   True)

_r("ckc_id_no_cross_ref", "linkage", "Ckc_Id Without Cross-Ref",
   "Contacts that carry a Ckc_Id in EQUIP but have no matching Registry cross-ref entry. "
   "Numerator: contacts where Ckc_Id is not null but cross_ref_number is absent. "
   "Denominator: all active non-employee contacts where Ckc_Id is not null.",
   True)

_r("type_mismatch_linkage", "linkage", "Linkage Type Mismatch",
   "Linked contacts where the linkage level conflicts with the contact type: "
   "C (Contact) types should link at entity level (contact_id = 0); "
   "B/I types should link at contact level (contact_id ≠ 0). "
   "Numerator: linked contacts with a type/level conflict. "
   "Denominator: all linked active contacts.",
   True)

_r("duplicate_entity_id", "linkage", "Duplicate Entity IDs",
   "Registry entity IDs linked to 2 or more distinct EQUIP contacts, "
   "indicating a one-to-many mapping that causes fan-out in joins. "
   "Numerator: entity IDs with 2+ linked EQUIP contacts. "
   "Denominator: all distinct entity IDs among linked contacts.",
   True)

_r("orphan_cross_ref", "linkage", "Orphan Cross-Ref Entries",
   "Hutson cross_ref entries whose contact code has no matching active EQUIP contact — "
   "the Registry linkage points to a contact that no longer exists. "
   "Numerator: Hutson cross_ref entries with no active EQUIP contact match. "
   "Denominator: all Hutson cross_ref entries.",
   True)

# ── Registry Parity — field-level (generated) ──────────────────────────────────
_parity_fields = [
    # (metric_prefix, label, scope_note, equip_field, registry_field)
    ("company_name",   "Company Name",    "B/C contacts only", "company_name",  "nm1_txt"),
    ("first_name",     "First Name",      "I/C contacts only", "name",          "first_nm"),
    ("last_name",      "Last Name",       "I/C contacts only", "surname",       "last_nm"),
    ("email",          "Email",           "All contact types", "email_address", "email_addr_txt"),
    ("business_phone", "Business Phone",  "All contact types", "BusinessPhone", "work_area_cd + work_phone_num"),
    ("private_phone",  "Home Phone",      "All contact types", "PrivatePhone",  "home_area_cd + home_phone_num"),
    ("mobile_phone",   "Mobile Phone",    "All contact types", "MobilePhone",   "mobile_area_cd + mobile_phone_num"),
    ("street",         "Street",          "All contact types", "street",        "phys_street1_txt"),
    ("city",           "City",            "All contact types", "city",          "phys_city"),
    ("state",          "State",           "All contact types", "state",         "phys_state_prov_cd"),
    ("zip",            "Zip Code",        "All contact types; US contacts compared on first 5 digits only", "pcode", "phys_postal_cd"),
    ("country",        "Country",         "All contact types (EQUIP side defaulted to US)", "country", "phys_iso2_cntry_cd"),
]
_parity_outcomes = [
    # (suffix, outcome_label, is_issue, outcome_desc)
    ("match",         "Match",          False, "EQUIP and Registry values agree (case-insensitive)"),
    ("mismatch",      "Mismatch",       True,  "EQUIP and Registry values conflict (case-insensitive)"),
    ("equip_only",    "EQUIP Only",     True,  "EQUIP has a value; Registry field is null"),
    ("registry_only", "Registry Only",  True,  "Registry has a value; EQUIP field is null"),
    ("both_null",     "Both Null",      False, "Both systems have no value for this field"),
]
for _pfx, _plbl, _scope, _efld, _rfld in _parity_fields:
    for _sfx, _olbl, _issue, _odesc in _parity_outcomes:
        _r(
            f"{_pfx}_{_sfx}", "parity", f"{_plbl} — {_olbl}",
            f"EQUIP {_efld} vs. Registry {_rfld}. {_odesc}. Scope: {_scope}. "
            f"Numerator: linked contacts in this parity state. "
            f"Denominator: all linked contacts in scope for this field.",
            _issue,
        )

# ── Registry Parity — priority metrics ────────────────────────────────────────
_r("phys_addr_certified_mismatch", "parity", "Certified Address Mismatch",
   "Linked contacts where the Registry holds a USPS-certified address that differs from EQUIP "
   "on at least one of street, city, state, or zip — high-confidence signal that EQUIP is out of date. "
   "Numerator: contacts with a certified Registry address that conflicts with EQUIP. "
   "Denominator: all linked contacts where phys_postal_certified = 'CERTIFIED'.",
   True)

_r("address_confirmed_undeliverable", "parity", "Confirmed Undeliverable Address",
   "Linked contacts where the Registry has flagged the physical or mailing address as undeliverable "
   "(phys_undeliverable_ind = 'Y' or mail_undeliverable_ind = 'Y'). "
   "Numerator: linked contacts with an undeliverable address flag. "
   "Denominator: all linked active contacts.",
   True)

_r("registry_oob_equip_active", "parity", "Registry OOB, EQUIP Active",
   "B contacts the Registry marks as out of business (out_of_busn_ind = 'Y') "
   "but EQUIP still shows as active — candidates for EQUIP inactivation review. "
   "Numerator: linked B contacts with Registry out_of_busn_ind = 'Y'. "
   "Denominator: all linked B contacts.",
   True)

_r("registry_deceased_equip_active", "parity", "Registry Deceased, EQUIP Active",
   "I/C contacts the Registry marks as deceased (descd_ind = 'Y') "
   "but EQUIP still shows as active — candidates for EQUIP inactivation review. "
   "Numerator: linked I/C contacts with Registry descd_ind = 'Y'. "
   "Denominator: all linked I/C contacts.",
   True)

_r("equip_inactive_reason_mismatch", "parity", "Inactive Reason Mismatch",
   "Inactive EQUIP contacts where Inactive_Reason conflicts with the Registry flag: "
   "'Out of Business' but out_of_busn_ind ≠ 'Y', or 'Deceased' but descd_ind ≠ 'Y'. "
   "Numerator: inactive EQUIP contacts with a conflicting reason/flag pair. "
   "Denominator: all linked inactive contacts.",
   True)

# ── Completeness ───────────────────────────────────────────────────────────────
_r("missing_first_name", "completeness", "Missing First Name",
   "I/C contacts with no first name (name field null or blank after trimming). "
   "Numerator: I/C contacts missing a first name. Denominator: all I/C active contacts.",
   True)
_r("missing_last_name", "completeness", "Missing Last Name",
   "I/C contacts with no last name (surname field null or blank after trimming). "
   "Numerator: I/C contacts missing a last name. Denominator: all I/C active contacts.",
   True)
_r("missing_company_name", "completeness", "Missing Company Name",
   "B/C contacts with no company name (company_name field null or blank after trimming). "
   "Numerator: B/C contacts missing a company name. Denominator: all B/C active contacts.",
   True)
_r("missing_street", "completeness", "Missing Street",
   "Contacts with no street address value. "
   "Numerator: contacts with a null/blank street. Denominator: all active contacts.",
   True)
_r("missing_city", "completeness", "Missing City",
   "Contacts with no city value. "
   "Numerator: contacts with a null/blank city. Denominator: all active contacts.",
   True)
_r("missing_state", "completeness", "Missing State",
   "Contacts with no state value. "
   "Numerator: contacts with a null/blank state. Denominator: all active contacts.",
   True)
_r("missing_zip", "completeness", "Missing Zip Code",
   "Contacts with no zip/postal code value. "
   "Numerator: contacts with a null/blank pcode. Denominator: all active contacts.",
   True)
_r("missing_country", "completeness", "Missing Country",
   "Contacts with no country value — raw null check, no US default applied here. "
   "Numerator: contacts with a null/blank country. Denominator: all active contacts.",
   True)
_r("missing_email", "completeness", "Missing Email",
   "Contacts with no email address. "
   "Numerator: contacts with a null/blank email_address. Denominator: all active contacts.",
   True)
_r("missing_all_phones", "completeness", "Missing All Phones",
   "Contacts where all three phone fields (BusinessPhone, PrivatePhone, MobilePhone) are null or blank. "
   "Numerator: contacts with no phone on record. Denominator: all active contacts.",
   True)
_r("no_contact_info", "completeness", "No Contact Info",
   "Contacts with no phone of any kind AND no email address — hardest segment to reach or match. "
   "Subset of missing_all_phones. "
   "Numerator: contacts with no phone and no email. Denominator: all active contacts.",
   True)

# ── Field Quality — Name / Company ─────────────────────────────────────────────
_r("placeholder_name", "field_quality", "Placeholder Name",
   "I/C contacts where the first or last name is a known placeholder "
   "(FIRSTNAME, FIRST NAME, FNAME, LASTNAME, LNAME, N/A, NA, NONE, UNKNOWN, NOT APPLICABLE, BLANK, etc.). "
   "Numerator: I/C contacts with a placeholder name value. Denominator: all I/C active contacts.",
   True)
_r("placeholder_company_name", "field_quality", "Placeholder Company Name",
   "B/C contacts where company_name is a known placeholder value "
   "(COMPANY, COMPANY NAME, N/A, NA, NONE, UNKNOWN, NOT APPLICABLE, BLANK, NO NAME, NAME, etc.). "
   "Numerator: B/C contacts with a placeholder company name. Denominator: all B/C active contacts.",
   True)
_r("name_all_same_char", "field_quality", "Repeated-Character Name",
   "I/C contacts where the first or last name is a single character repeated (e.g., XXXXX, .....). "
   "Numerator: I/C contacts with a repeated-character name. Denominator: all I/C active contacts.",
   True)
_r("company_name_all_same_char", "field_quality", "Repeated-Character Company Name",
   "B/C contacts where company_name is a single character repeated (e.g., XXXXX, .....). "
   "Numerator: B/C contacts with a repeated-character company name. Denominator: all B/C active contacts.",
   True)
_r("name_numeric_only", "field_quality", "Numeric-Only Name",
   "I/C contacts where the first or last name contains only digits. "
   "Numerator: I/C contacts with a numeric-only name. Denominator: all I/C active contacts.",
   True)
_r("company_name_numeric_only", "field_quality", "Numeric-Only Company Name",
   "B/C contacts where company_name contains only digits. "
   "Numerator: B/C contacts with a numeric-only company name. Denominator: all B/C active contacts.",
   True)
_r("status_text_in_name", "field_quality", "Status Text in Name",
   "I/C contacts where first or last name contains alert/status keywords "
   "(DECEASED, OUT OF BUSINESS, DO NOT USE, INACTIVE, CLOSED, FARM PLAN, BAD DEBT, WRITE OFF, DO NOT SELL, DO NOT SERVE). "
   "Numerator: I/C contacts with status text in a name field. Denominator: all I/C active contacts.",
   True)
_r("status_text_in_company", "field_quality", "Status Text in Company Name",
   "B/C contacts where company_name contains alert/status keywords "
   "(DECEASED, OUT OF BUSINESS, DO NOT USE, INACTIVE, CLOSED, OOB, BAD DEBT, WRITE OFF, DO NOT SELL, DO NOT SERVE, etc.). "
   "Numerator: B/C contacts with status text in company_name. Denominator: all B/C active contacts.",
   True)
_r("status_text_in_street", "field_quality", "Status Text in Street",
   "Contacts where the street field contains alert/status keywords "
   "(DECEASED, OUT OF BUSINESS, DO NOT USE, INACTIVE, CLOSED, BAD DEBT, WRITE OFF, DO NOT SELL, DO NOT SERVE). "
   "Numerator: contacts with status text in the street field. Denominator: all active contacts.",
   True)
_r("dba_in_company_name", "field_quality", "DBA Pattern in Company Name",
   "B/C contacts where company_name contains a DBA pattern (DBA, D/B/A, DOING BUSINESS AS). "
   "Data belongs in the Doing_Business_As field. "
   "Numerator: B/C contacts with a DBA pattern in company_name. Denominator: all B/C active contacts.",
   True)
_r("test_record", "field_quality", "Test / Dummy Record",
   "Contacts where a name or company field matches known test-record values "
   "(TEST, TESTING, TEMP, DUMMY, SAMPLE). "
   "Numerator: contacts that appear to be test records. Denominator: all active contacts.",
   True)
_r("contact_type_field_mismatch", "field_quality", "Contact Type Field Mismatch",
   "Contacts where name fields are inconsistent with the Business_Individual type: "
   "a B contact with a personal name but no company, or an I/C contact with a company but no personal name. "
   "Numerator: contacts with a type/field conflict. Denominator: all active contacts.",
   True)
_r("prefix_in_name", "field_quality", "Prefix in First Name",
   "I/C contacts where the first name starts with a title prefix (Mr., Mrs., Ms., Dr., Rev., Prof., etc.). "
   "Data belongs in the title field. "
   "Numerator: I/C contacts with a prefix in the first name. Denominator: all I/C active contacts.",
   True)
_r("suffix_in_surname", "field_quality", "Suffix in Last Name",
   "I/C contacts where the last name ends with a generation or credential suffix (JR, SR, II, MD, PHD, etc.). "
   "Data belongs in the Generation or Suffix field. "
   "Numerator: I/C contacts with a suffix in the last name. Denominator: all I/C active contacts.",
   True)
_r("combined_names_in_name", "field_quality", "Combined Names in First Name",
   "I/C contacts where the first name contains '&', 'AND', '/', or 'OR' — "
   "likely two people stored in one record. "
   "Numerator: I/C contacts with combined names. Denominator: all I/C active contacts.",
   True)
_r("familiar_name_pattern", "field_quality", "Familiar Name in Parentheses",
   "I/C contacts where the first name contains a parenthetical (e.g., 'Billy (Joe)'). "
   "Common data-entry shorthand; not invalid but worth reviewing. "
   "Numerator: I/C contacts with this pattern. Denominator: all I/C active contacts.",
   True)

# ── Field Quality — Email ──────────────────────────────────────────────────────
_r("email_invalid_format", "field_quality", "Invalid Email Format",
   "Contacts with an email address that doesn't match the expected pattern (must contain '@' followed by a domain with '.'). "
   "Numerator: contacts with a non-null email that fails the format check. "
   "Denominator: all contacts with a non-null email address.",
   True)
_r("email_placeholder", "field_quality", "Placeholder Email",
   "Contacts where the email matches a known placeholder pattern "
   "(noemail@..., test@test..., none@none..., noreply@..., donotcontact@...). "
   "Numerator: contacts with a placeholder email. "
   "Denominator: all contacts with a non-null email address.",
   True)

# ── Field Quality — Phone (generated) ─────────────────────────────────────────
for _ph_pfx, _ph_lbl, _equip_fld in [
    ("biz_phone",  "Business Phone", "BusinessPhone"),
    ("priv_phone", "Home Phone",     "PrivatePhone"),
    ("mob_phone",  "Mobile Phone",   "MobilePhone"),
]:
    _r(f"{_ph_pfx}_sequential", "field_quality", f"{_ph_lbl} — Sequential Digits",
       f"Contacts where {_equip_fld} is the sequential placeholder '1234567890'. "
       f"Numerator: contacts with this value. "
       f"Denominator: all contacts with a non-null {_equip_fld}.",
       True)
    _r(f"{_ph_pfx}_repeated_digit", "field_quality", f"{_ph_lbl} — Repeated Digit",
       f"Contacts where {_equip_fld} is a 10-digit number composed of one repeated digit (e.g., 0000000000). "
       f"Numerator: contacts with an all-same-digit phone. "
       f"Denominator: all contacts with a non-null {_equip_fld}.",
       True)
    _r(f"{_ph_pfx}_wrong_length", "field_quality", f"{_ph_lbl} — Wrong Length",
       f"Contacts where {_equip_fld} is not 10 or 11 digits (11-digit valid if prefixed with country code 1). "
       f"Numerator: contacts with an invalid-length phone number. "
       f"Denominator: all contacts with a non-null {_equip_fld}.",
       True)

# ── Field Quality — Address Format ────────────────────────────────────────────
_r("placeholder_street", "field_quality", "Placeholder Street",
   "Contacts where the street field is a known placeholder value (N/A, NONE, UNKNOWN, NO ADDRESS, ADDRESS, -). "
   "Numerator: contacts with a placeholder street value. Denominator: all contacts with a non-null street.",
   True)
_r("placeholder_city", "field_quality", "Placeholder City",
   "Contacts where the city field is a known placeholder value (N/A, NONE, UNKNOWN, NO CITY, -). "
   "Numerator: contacts with a placeholder city value. Denominator: all contacts with a non-null city.",
   True)
_r("placeholder_state", "field_quality", "Placeholder State",
   "Contacts where the state field is a known placeholder value (N/A, NONE, UNKNOWN, XX, ZZ). "
   "Numerator: contacts with a placeholder state value. Denominator: all contacts with a non-null state.",
   True)
_r("state_not_2char", "field_quality", "State Not 2 Characters",
   "Contacts where the state field is non-null but not exactly 2 characters — catches written-out state names. "
   "Numerator: contacts with a non-2-char state. Denominator: all contacts with a non-null state.",
   True)
_r("country_not_2char", "field_quality", "Country Not 2 Characters",
   "Contacts where the country field is non-null but not exactly 2 characters "
   "(catches UNITED STATES, USA, etc. — should be the ISO-2 code 'US'). "
   "Numerator: contacts with a non-2-char country. Denominator: all contacts with a non-null country.",
   True)
_r("zip_invalid_format", "field_quality", "Invalid Zip Format",
   "US contacts where the zip code doesn't match any valid US format: "
   "5-digit (12345), ZIP+4 with dash (12345-6789), or 9-digit no-dash (123456789). "
   "Numerator: US contacts with a non-null zip that fails all three format checks. "
   "Denominator: all US contacts with a non-null zip code.",
   True)

# ── Field Quality — Coded Fields ──────────────────────────────────────────────
_r("generation_unrecognized", "field_quality", "Unrecognized Generation",
   "I/C contacts where the Generation field is outside the valid list (Jr, Jr., Junior, Sr, Sr., Senior, II, III, IV, V). "
   "Numerator: I/C contacts with an unrecognized Generation value. "
   "Denominator: all I/C contacts with a non-null Generation.",
   True)
_r("title_unrecognized", "field_quality", "Unrecognized Prefix (title)",
   "I/C contacts where the title field is outside the valid prefix list "
   "(Mr, Mrs, Ms, Miss, Dr, Rev, Prof, Capt, Sgt, Col, Maj, Gen, Hon, etc.). "
   "Numerator: I/C contacts with an unrecognized title. "
   "Denominator: all I/C contacts with a non-null title.",
   True)
_r("suffix_unrecognized", "field_quality", "Unrecognized Suffix",
   "I/C contacts where the Suffix field is outside the valid credential list "
   "(MD, DO, DDS, DMD, DVM, PhD, JD, CPA, RN, NP, PA, PE, Esq, MBA, CFA, CFP, LCSW, etc.). "
   "Numerator: I/C contacts with an unrecognized suffix. "
   "Denominator: all I/C contacts with a non-null Suffix.",
   True)

# ── Field Quality — Contact Code Integrity ────────────────────────────────────
_r("invalid_branch", "field_quality", "Invalid Branch Code",
   "Contacts assigned to a known closed or invalid TERRITORY code (07, 13, 52, 55, 61, 63, 64, 67, 70, 71). "
   "Numerator: contacts on a closed branch code. "
   "Denominator: all contacts with a non-null branch.",
   True)
_r("contact_code_duplicate_normalized", "field_quality", "Duplicate Contact Code (Normalized)",
   "Active contacts whose contact_code, after uppercasing and trimming, matches another active contact's code. "
   "Both the original and the duplicate are counted in the numerator. "
   "Numerator: contacts in a duplicated-code group. Denominator: all active contacts in that dimension slice.",
   True)
_r("contact_code_has_whitespace", "field_quality", "Contact Code Has Whitespace",
   "Contacts where the contact_code contains leading, trailing, or embedded whitespace, "
   "which causes cross_ref join failures. "
   "Numerator: contacts with whitespace in their code. Denominator: all active contacts.",
   True)
_r("contact_code_non_alphanumeric", "field_quality", "Contact Code Has Non-Alphanumeric Characters",
   "Contacts where the contact_code contains characters outside A–Z and 0–9. "
   "Superset of the whitespace check — both tracked separately to distinguish failure types. "
   "Numerator: contacts with non-alphanumeric characters. Denominator: all active contacts.",
   True)

# ── Staleness ──────────────────────────────────────────────────────────────────
# Informational distribution — all active contacts fall into exactly one bucket.
# Numerators across all buckets sum to the shared denominator within each dimension group.
_staleness = [
    ("no_account",       "No Account",
     "Active non-employee contacts with no ArMaster_Customer record — "
     "may be data-entry orphans or structurally expected for some contact types. "
     "Numerator: contacts with no account record. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("never_transacted",  "Never Transacted",
     "Contacts with an ArMaster record but no transaction history in the 5-year revenue window. "
     "Different from no_account: the account exists but was never used. "
     "Numerator: contacts with an account but no transactions. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("0-1yr",  "Active — Last Year",
     "Last transaction within the past 12 months — healthy active customer baseline. "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("1-2yr",  "Lapsing — 1–2 Years",
     "Last transaction 1–2 years ago. "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("2-3yr",  "Lapsing — 2–3 Years",
     "Last transaction 2–3 years ago. "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("3-4yr",  "At Risk — 3–4 Years",
     "Last transaction 3–4 years ago. "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("4-5yr",  "At Risk — 4–5 Years",
     "Last transaction 4–5 years ago — approaching the 5-year inactivation threshold. "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
    ("5+yr",   "Stale — 5+ Years",
     "Last transaction more than 5 years ago — primary inactivation candidates (Phase 2.1). "
     "Numerator: contacts in this staleness bucket. "
     "Denominator: all active contacts in the same decile/branch/cohort."),
]
for _mn, _lbl, _defn in _staleness:
    _r(_mn, "staleness", _lbl, _defn, False)

# ── Match Readiness ────────────────────────────────────────────────────────────
# Informational distribution — all unlinked active contacts fall into exactly one tier.
_r("tier_1_strong", "match_readiness", "Tier 1 — Strong Match Candidate",
   "Unlinked contacts with a usable name AND a full or zip-anchored address "
   "(street + city + state, OR street + zip). Tight Registry match likely. "
   "Numerator: unlinked contacts in this tier. "
   "Denominator: all unlinked active contacts in the same dimension slice.",
   False)
_r("tier_2_partial", "match_readiness", "Tier 2 — Partial Match Candidate",
   "Unlinked contacts with a usable name AND partial address or at least one contact method (phone or email). "
   "Match possible depending on Registry data quality. "
   "Numerator: unlinked contacts in this tier. "
   "Denominator: all unlinked active contacts in the same dimension slice.",
   False)
_r("tier_3_name_only", "match_readiness", "Tier 3 — Name Only",
   "Unlinked contacts with a usable name but no address fields and no contact info. "
   "Unlikely to match without data enrichment. "
   "Numerator: unlinked contacts in this tier. "
   "Denominator: all unlinked active contacts in the same dimension slice.",
   False)
_r("tier_4_no_name", "match_readiness", "Tier 4 — No Name",
   "Unlinked contacts missing the primary name field(s) for their contact type "
   "(first/last for I/C; company_name for B). Cannot attempt a Registry match regardless of other fields. "
   "Numerator: unlinked contacts with no usable name. "
   "Denominator: all unlinked active contacts in the same dimension slice.",
   False)

# ── Build DataFrame and write ──────────────────────────────────────────────────
_dim_pdf = pd.DataFrame(_rows)
_dim_sdf = spark.createDataFrame(_dim_pdf)

(
    _dim_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_LAKEHOUSE}.{DIM_TABLE}")
)

print(f"Written {len(_rows)} rows to {TARGET_LAKEHOUSE}.{DIM_TABLE}")
_dim_pdf.groupby(["metric_category", "is_issue"])["metric_name"].count().rename("count").reset_index()


In [ ]:
# ── Validation — Fact / Dimension Metric Name Parity ──────────────────────────
# Confirms exact overlap: every metric_name the snapshot produces must exist in
# the dim table, and every dim table row must correspond to a metric the snapshot
# actually produces. No extras, no missing on either side.

_fact_names = sdf.select("metric_name").distinct()
_dim_names  = _dim_sdf.select("metric_name").distinct()

_missing_from_dim = _fact_names.subtract(_dim_names)   # in snapshot, absent from dim
_extra_in_dim     = _dim_names.subtract(_fact_names)   # in dim, not produced by snapshot

_n_missing = _missing_from_dim.count()
_n_extra   = _extra_in_dim.count()

if _n_missing:
    print(f"FAIL — {_n_missing} metric(s) produced by snapshot but absent from dim table:")
    _missing_from_dim.orderBy("metric_name").show(200, truncate=False)

if _n_extra:
    print(f"FAIL — {_n_extra} metric(s) in dim table but not produced by snapshot:")
    _extra_in_dim.orderBy("metric_name").show(200, truncate=False)

if _n_missing == 0 and _n_extra == 0:
    _total = _fact_names.count()
    print(f"PASS — {_total} metric names match exactly between snapshot and dim table.")
else:
    raise ValueError(
        f"Metric name mismatch: {_n_missing} missing from dim, {_n_extra} extra in dim."
    )
